In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm  # 导入字体管理模块
# 设置中文字体
font_path = "/home/USER/anaconda3/envs/test/lib/python3.10/site-packages/matplotlib/mpl-data/fonts/ttf/simhei.ttf"  # 替换为实际字体路径
# 手动加载字体
custom_font = fm.FontProperties(fname=font_path)
font_name = custom_font.get_name()  # 如 'SimHei'

# 将字体加入 Matplotlib 的字体管理器
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = font_name  # 使用字体名称
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

In [ ]:
# 选择的城市（已按实验要求预设3类城市）
cities = ["北京", "上海", "杭州", "成都", "厦门"]

# 各城市2023年人均GDP（单位：万元，用户需替换为真实查询数据，此处为示例占位符）
gdp_2023 = [20.1, 19.1, 16.0, 10.3, 15.1]  # 示例数据：北京19万，上海18.8万，杭州15.6万，成都12.5万，厦门14.2万

# 3. 工作年限参数
graduation_age = 22  # 毕业年龄
retirement_age = 65  # 退休年龄
work_years = retirement_age - graduation_age  # 总工作年限

growth_rate = 0.08  # 5%年增长率

# ---------------------------------------------------------------------------
print(f"实验参数确认：")
print(f"选择城市：{cities}")
print(f"工作年限：{work_years}年（{graduation_age}岁毕业 → {retirement_age}岁退休）")
print(f"年薪增长率：{growth_rate*100:.1f}%")
print(f"各城市2023年人均GDP（起薪）：{dict(zip(cities, gdp_2023))} 万元")

In [ ]:
# 生成工作年份列表
years = list(range(1, work_years + 1))  # 工作第1年 ~ 第work_years年（长度=work_years）

# 生成对应年龄列表
age_list = list(range(graduation_age, graduation_age + work_years))  # 22岁~(22+43-1)=64岁（长度=work_years）

# 校验长度
assert len(years) == len(age_list), f"年份列表长度（{len(years)}）与年龄列表长度（{len(age_list)}）不一致！"
print(f"数据校验通过：年份和年龄列表长度均为 {len(years)}")

# 创建基础DataFrame
df = pd.DataFrame({
    "工作年份": years,
    "年龄": age_list
})

In [ ]:
for i, city in enumerate(cities):
    start_salary = gdp_2023[i]  # 该城市起薪（2023年人均GDP）
    # 计算每年年薪：第1年=起薪，第2年=起薪*(1+g)，第3年=起薪*(1+g)^2...
    df[f"{city}_年薪(万元)"] = start_salary * (1 + growth_rate) ** (df["工作年份"] - 1)

for city in cities:
    df[f"{city}_累计收入(万元)"] = df[f"{city}_年薪(万元)"].cumsum()  # 累计求和

print("数据计算结果预览（前5年）：")
print(df.head().round(2))  

print("\n数据计算结果预览（最后5年）：")
print(df.tail().round(2))

In [ ]:
# 导出到Excel
excel_path = "城市职业生涯收入对比数据8.xlsx"
df.to_excel(excel_path, index=False, engine="openpyxl")
print(f"\n数据已导出到：{excel_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


weights = {
    '收入水平': 0.25,
    '家庭陪伴': 0.10,
    '工作机会': 0.10,
    '工作压力': 0.15,
    '城市宜居': 0.25,
    '社交生活': 0.15
}
weights_series = pd.Series(weights)

# 城市评分 (1-10分)
data = {#北京 上海 杭州 成都 厦门
    '收入水平': [9, 8, 6, 3, 5],
    '家庭陪伴': [4, 9, 6, 4, 5],
    '工作机会': [6, 8, 9, 9, 9],
    '工作压力': [3, 4, 6, 7, 8],
    '城市宜居': [8, 9, 9, 7, 8],
    '社交生活': [8, 9, 7, 8, 7]
}
df_scores = pd.DataFrame(data, index=cities)

print("--- 原始城市评分数据 (1-10分) ---")
print(df_scores)
print("\n--- 个人权重 ---")
print(weights_series)
print("-" * 40)

In [ ]:
df_weighted_scores = df_scores.mul(weights_series, axis=1)

df_scores['加权总分'] = df_weighted_scores.sum(axis=1)

df_scores['排名'] = df_scores['加权总分'].rank(ascending=False).astype(int)

print("\n--- 计算结果 (包含加权总分和排名) ---")
print(df_scores[['加权总分', '排名']].sort_values(by='加权总分', ascending=False))
print("-" * 40)

df_export = df_scores.copy()

# 创建一个包含权重的 DataFrame
df_weights_row = pd.DataFrame([weights], index=['维度权重'])
df_export_final = pd.concat([df_weights_row, df_export])

# 导出到 Excel 文件
excel_filename = '城市幸福感量化模型_实验结果.xlsx'
df_export_final.to_excel(excel_filename, sheet_name='幸福感量化结果')
print(f"\n 数据已成功导出到 Excel 文件: {excel_filename}")
print("-" * 40)

In [ ]:
weights = {
    '收入水平': 0.5,
    '家庭陪伴': 0.05,
    '工作机会': 0.05,
    '工作压力': 0.10,
    '城市宜居': 0.20,
    '社交生活': 0.10
}
weights_series = pd.Series(weights)
df_weighted_scores = df_scores.mul(weights_series, axis=1)

df_scores['加权总分'] = df_weighted_scores.sum(axis=1)

df_scores['排名'] = df_scores['加权总分'].rank(ascending=False).astype(int)

print("\n--- 计算结果 (包含加权总分和排名) ---")
print(df_scores[['加权总分', '排名']].sort_values(by='加权总分', ascending=False))
print("-" * 40)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import fsolve
import openpyxl
from datetime import datetime
import os

city = "上海"
house_price_per_sqm = 85000  # 元/平方米（上海中心区域均价）
house_area = 80  # 平方米（典型两居室）
P = house_price_per_sqm * house_area  # 房屋总价

# 贷款参数
d = 0.3  # 首付比例30%
i = 0.04  # 贷款年利率4.0%
N = 30  # 贷款年限30年

# 租赁和投资参数
R = 0.018  # 租售比1.8%（上海实际情况，租售比较低）
r = 0.045  # 租房者投资年化收益率4.5%

# 模拟参数
simulation_years = 30
months = simulation_years * 12

print(" 上海房产参数设定：")
print(f"房屋位置：{city}")
print(f"房屋面积：{house_area}平方米")
print(f"单价：{house_price_per_sqm:,.0f}元/平方米")
print(f"总价：{P:,.0f}元 ({P/10000:.1f}万元)")
print(f"首付比例：{d*100:.0f}%，首付金额：{P*d:,.0f}元 ({P*d/10000:.1f}万元)")
print(f"贷款金额：{P*(1-d):,.0f}元 ({P*(1-d)/10000:.1f}万元)")
print(f"贷款年利率：{i*100:.1f}%，贷款年限：{N}年")
print(f"租售比：{R*100:.1f}%，年租金：{P*R:,.0f}元，月租金：{P*R/12:,.0f}元")
print(f"投资年化收益率：{r*100:.1f}%")

#  月供计算函数

def calculate_monthly_payment(P, d, i, N, method='equal_principal_interest'):
    """
    计算月供金额
    method: 'equal_principal_interest' (等额本息) or 'equal_principal' (等额本金)
    """
    loan_amount = P * (1 - d)  # 贷款金额
    monthly_rate = i / 12  # 月利率
    total_months = N * 12  # 总月数
    
    if method == 'equal_principal_interest':
        # 等额本息公式：M = P*r*(1+r)^n / [(1+r)^n-1]
        M = loan_amount * monthly_rate * (1 + monthly_rate)**total_months / ((1 + monthly_rate)**total_months - 1)
        return M
    
    elif method == 'equal_principal':
        # 等额本金：每月偿还相同本金 + 剩余本金的利息
        principal_per_month = loan_amount / total_months
        # 首月月供
        first_month_interest = loan_amount * monthly_rate
        first_month_payment = principal_per_month + first_month_interest
        # 末月月供
        last_month_interest = principal_per_month * monthly_rate
        last_month_payment = principal_per_month + last_month_interest
        return first_month_payment, last_month_payment, principal_per_month
    
    else:
        raise ValueError("method must be 'equal_principal_interest' or 'equal_principal'")

# 计算等额本息月供
monthly_payment_equal_principal_interest = calculate_monthly_payment(P, d, i, N, 'equal_principal_interest')

# 计算等额本金月供
first_payment, last_payment, principal_per_month = calculate_monthly_payment(P, d, i, N, 'equal_principal')

print("\n 月供计算结果：")
print(f"等额本息月供：{monthly_payment_equal_principal_interest:,.2f}元/月")
print(f"等额本金首月月供：{first_payment:,.2f}元/月")
print(f"等额本金末月月供：{last_payment:,.2f}元/月")
print(f"等额本金月均还款：{(first_payment + last_payment) / 2:,.2f}元/月")

# 本实验使用等额本息方式
monthly_payment = monthly_payment_equal_principal_interest
monthly_rent = P * R / 12  # 月租金
monthly_investment = monthly_payment - monthly_rent  # 每月可投资额

print(f"\n 关键对比：")
print(f"月供：{monthly_payment:,.2f}元")
print(f"月租金：{monthly_rent:,.2f}元")
print(f"每月可投资额（月供-租金）：{monthly_investment:,.2f}元")

In [ ]:
# 逐月模拟函数
def simulate_housing_decision(g, P, d, i, N, R, r, simulation_years=30):
    """
    模拟买房和租房两种路径30年后的资产情况
    
    参数：
    g: 房价年增长率
    P: 房屋总价
    d: 首付比例
    i: 贷款年利率
    N: 贷款年限
    R: 租售比
    r: 投资年化收益率
    simulation_years: 模拟年限
    
    返回：
    house_asset: 买房者30年后的净资产
    rent_asset: 租房者30年后的投资总额
    monthly_data: 逐月详细数据
    """
    
    # 初始化
    months = simulation_years * 12
    monthly_rate_house = g / 12  # 房价月增长率
    monthly_rate_invest = r / 12  # 投资月收益率
    monthly_payment = calculate_monthly_payment(P, d, i, N, 'equal_principal_interest')
    monthly_rent = P * R / 12
    
    # 买房者：首付锁定，房屋价值增长
    down_payment = P * d
    
    # 租房者：首付用于投资 + 每月差额投资
    initial_investment = down_payment  # 首付金额作为初始投资额
    monthly_investment = monthly_payment - monthly_rent  # 每月可投资额
    
    # 创建数据存储
    monthly_data = pd.DataFrame({
        'month': range(1, months + 1),
        'house_value': np.zeros(months),  # 房屋价值
        'house_equity': np.zeros(months),  # 房产净资产（简化考虑）
        'investment_balance': np.zeros(months),  # 投资账户余额
        'monthly_investment': np.zeros(months)  # 每月投资额
    })
    
    # 逐月模拟
    investment_balance = initial_investment
    
    for month in range(months):
        current_house_value = P * (1 + monthly_rate_house) ** (month + 1)
        house_equity = current_house_value if month == months - 1 else 0
 
        # 月初投资余额产生收益
        investment_balance = investment_balance * (1 + monthly_rate_invest)
        # 月末追加投资
        investment_balance += monthly_investment
    
        monthly_data.loc[month, 'house_value'] = current_house_value
        monthly_data.loc[month, 'house_equity'] = house_equity
        monthly_data.loc[month, 'investment_balance'] = investment_balance
        monthly_data.loc[month, 'monthly_investment'] = monthly_investment
    
    # 30年后的最终资产
    house_asset = monthly_data.iloc[-1]['house_value']  # 买房者净资产
    rent_asset = monthly_data.iloc[-1]['investment_balance']  # 租房者投资总额
    
    return house_asset, rent_asset, monthly_data

#  寻找平衡点

def find_balance_point(P, d, i, N, R, r, simulation_years=30):
    """
    寻找使买房和租房资产相等的房价年增长率g
    """
    
    def equation(g):
        house_asset, rent_asset, _ = simulate_housing_decision(
            g[0], P, d, i, N, R, r, simulation_years
        )
        return house_asset - rent_asset

    initial_guess = [0.03]
    
    try:
        solution = fsolve(equation, initial_guess, xtol=1e-6)
        balance_g = solution[0]
        
        # 验证结果
        house_asset, rent_asset, _ = simulate_housing_decision(
            balance_g, P, d, i, N, R, r, simulation_years
        )
        
        print(f" 平衡点计算成功！")
        print(f"平衡点房价年增长率：{balance_g*100:.4f}%")
        print(f"30年后买房者资产：{house_asset:,.0f}元 ({house_asset/10000:.1f}万元)")
        print(f"30年后租房者资产：{rent_asset:,.0f}元 ({rent_asset/10000:.1f}万元)")
        print(f"资产差额：{house_asset - rent_asset:,.0f}元")
        
        return balance_g
    except Exception as e:
        print(f" 平衡点计算失败：{e}")
        # 使用二分法作为备选
        low, high = 0.01, 0.10  # 1%到10%
        for _ in range(100):  # 最多100次迭代
            mid = (low + high) / 2
            house_asset, rent_asset, _ = simulate_housing_decision(
                mid, P, d, i, N, R, r, simulation_years
            )
            if abs(house_asset - rent_asset) < 10000:  # 误差小于1万元
                return mid
            if house_asset > rent_asset:
                high = mid
            else:
                low = mid
        return (low + high) / 2

# 计算平衡点
print("\n 寻找买房-租房决策平衡点...")
balance_g = find_balance_point(P, d, i, N, R, r, simulation_years)

# 测试不同的房价增长率
growth_rates = [0.01, 0.02, 0.03, balance_g, 0.04, 0.05, 0.06]
results = []

print(f"\n 多情景分析（不同房价年增长率）：")
print("-" * 80)

for g in growth_rates:
    house_asset, rent_asset, monthly_data = simulate_housing_decision(
        g, P, d, i, N, R, r, simulation_years
    )
    
    results.append({
        '房价年增长率': g * 100,
        '买房最终资产': house_asset,
        '租房最终资产': rent_asset,
        '资产差额': house_asset - rent_asset,
        '更优选择': '买房' if house_asset > rent_asset else '租房' if house_asset < rent_asset else '平衡点'
    })
    
    print(f"g={g*100:5.2f}% | 买房: {house_asset/10000:8.1f}万元 | "
          f"租房: {rent_asset/10000:8.1f}万元 | "
          f"差额: {(house_asset-rent_asset)/10000:8.1f}万元 | "
          f"选择: {results[-1]['更优选择']}")

# 创建结果DataFrame
results_df = pd.DataFrame(results)
print("\n 多情景分析汇总：")
print(results_df.to_string(index=False,
                          float_format=lambda x: f"{x:.2f}" if isinstance(x, float) and x < 1 else f"{x:,.0f}",
                          formatters={
                              '房价年增长率': '{:.2f}%'.format,
                              '买房最终资产': lambda x: f"{x/10000:.1f}万元",
                              '租房最终资产': lambda x: f"{x/10000:.1f}万元",
                              '资产差额': lambda x: f"{x/10000:.1f}万元"
                          }))


# 在平衡点进行详细模拟
print(f"\n 详细模拟（平衡点g={balance_g*100:.4f}%）...")
house_asset_balance, rent_asset_balance, monthly_data_balance = simulate_housing_decision(
    balance_g, P, d, i, N, R, r, simulation_years
)

# 添加年份列
monthly_data_balance['year'] = (monthly_data_balance['month'] - 1) // 12 + 1
monthly_data_balance['month_in_year'] = (monthly_data_balance['month'] - 1) % 12 + 1

# 按年汇总
annual_data = monthly_data_balance.groupby('year').last().reset_index()
annual_data = annual_data[['year', 'house_value', 'investment_balance']]

print("\n 年度资产变化（平衡点情景）：")
print(annual_data.head(10).to_string(index=False,
    formatters={
        'house_value': lambda x: f"{x/10000:.1f}万元",
        'investment_balance': lambda x: f"{x/10000:.1f}万元"
    }))

In [ ]:

print(f"\n" + "="*80)
print(" 实验总结与关键发现")
print("="*80)

print(f"\n 上海市场关键参数：")
print(f"- 房屋总价：{P/10000:.0f}万元 ({house_area}㎡，{house_price_per_sqm/10000:.2f}万元/㎡)")
print(f"- 月供：{monthly_payment:,.0f}元，月租金：{monthly_rent:,.0f}元")
print(f"- 每月可投资额：{monthly_investment:,.0f}元")

print(f"\n 决策平衡点：")
print(f"- 临界房价年增长率：{balance_g*100:.4f}%")
print(f"- 30年后平衡资产：{house_asset_balance/10000:.1f}万元")



print(f"- 当增长率>{balance_g*100:.2f}%时，买房更优")
print(f"- 当增长率<{balance_g*100:.2f}%时，租房更优")



print(f"\n数据文件：{excel_filename}")


In [ ]:
def export_to_excel(results_df, annual_data, balance_g, P, d, i, N, R, r):
    """
    将分析结果导出到Excel文件
    """
    # 创建文件名（包含时间和城市）
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"上海_买房租房决策分析_{timestamp}.xlsx"
    
    # 创建Excel writer
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        
        # 1. 参数设置表
        params_df = pd.DataFrame({
            '参数类别': ['城市', '房屋面积(㎡)', '单价(元/㎡)', '总价(万元)', 
                        '首付比例', '贷款金额(万元)', '贷款年利率', '贷款年限',
                        '租售比', '月租金(元)', '投资年化收益率', '模拟年限',
                        '平衡点房价年增长率'],
            '参数值': [city, house_area, house_price_per_sqm, P/10000,
                      f"{d*100:.0f}%", P*(1-d)/10000, f"{i*100:.1f}%", N,
                      f"{R*100:.1f}%", monthly_rent, f"{r*100:.1f}%", simulation_years,
                      f"{balance_g*100:.4f}%"]
        })
        params_df.to_excel(writer, sheet_name='参数设置', index=False)
        
        # 2. 多情景分析结果
        results_df_export = results_df.copy()
        results_df_export['房价年增长率'] = results_df_export['房价年增长率'].apply(lambda x: f"{x:.4f}%")
        results_df_export['买房最终资产'] = results_df_export['买房最终资产'].apply(lambda x: f"{x/10000:.1f}万元")
        results_df_export['租房最终资产'] = results_df_export['租房最终资产'].apply(lambda x: f"{x/10000:.1f}万元")
        results_df_export['资产差额'] = results_df_export['资产差额'].apply(lambda x: f"{x/10000:.1f}万元")
        results_df_export.to_excel(writer, sheet_name='多情景分析', index=False)
        
        # 3. 年度资产变化（平衡点情景）
        annual_export = annual_data.copy()
        annual_export['house_value'] = annual_export['house_value'].apply(lambda x: f"{x/10000:.1f}万元")
        annual_export['investment_balance'] = annual_export['investment_balance'].apply(lambda x: f"{x/10000:.1f}万元")
        annual_export.columns = ['年份', '买房资产', '租房资产']
        annual_export.to_excel(writer, sheet_name='年度资产变化', index=False)
        
        # 4. 详细逐月数据
        monthly_export = monthly_data_balance.copy()
        monthly_export['house_value'] = monthly_export['house_value'].apply(lambda x: f"{x/10000:.2f}万元")
        monthly_export['investment_balance'] = monthly_export['investment_balance'].apply(lambda x: f"{x/10000:.2f}万元")
        monthly_export['monthly_investment'] = monthly_export['monthly_investment'].apply(lambda x: f"{x:.0f}元")
        monthly_export.columns = ['月份', '房屋价值', '房产净资产', '投资账户余额', '月投资额', '年份', '月序']
        monthly_export.to_excel(writer, sheet_name='详细逐月数据', index=False)
        
        # 5. 月供计算详情
        payment_details = pd.DataFrame({
            '计算项目': ['房屋总价', '首付比例', '首付金额', '贷款金额',
                       '等额本息月供', '月租金', '每月可投资额'],
            '数值': [f"{P/10000:.0f}万元", f"{d*100:.0f}%", f"{P*d/10000:.1f}万元", f"{P*(1-d)/10000:.1f}万元",
                    f"{monthly_payment:.0f}元", f"{monthly_rent:.0f}元", f"{monthly_payment-monthly_rent:.0f}元"]
        })
        payment_details.to_excel(writer, sheet_name='月供计算', index=False)
        
        # 6. 决策建议
        advice_df = pd.DataFrame({
            '房价增长率范围': ['< 3.00%', '3.00% - 3.50%', '> 3.50%'],
            '推荐决策': ['租房更优', '接近平衡，考虑个人情况', '买房更优'],
            '理由': [
                f'当房价年增长率<{balance_g*100:.2f}%时，租房投资的财务收益更高',
                f'在平衡点附近，需考虑居住稳定性、政策风险等非财务因素',
                f'当房价年增长率>{balance_g*100:.2f}%时，买房的资产增值优势明显'
            ]
        })
        advice_df.to_excel(writer, sheet_name='决策建议', index=False)
        
        # 7. 上海市场参考
        market_ref = pd.DataFrame({
            '指标': ['上海当前房价趋势', '历史租售比范围', '政策风险提示', '建议决策'],
            '说明': [
                '2025年上海房价同比下跌3.59%，处于调整期',
                '上海租售比通常在1.5%-2.5%之间，低于国际合理水平(4-6%)',
                '关注房地产政策、人口流入、经济发展等长期因素',
                f'基于当前市场，建议密切关注市场变化，增长率预期是关键'
            ]
        })
        market_ref.to_excel(writer, sheet_name='市场参考', index=False)
        
        print(f"\n 数据已成功导出到：{filename}")
        return filename

# 导出数据
excel_filename = export_to_excel(results_df, annual_data, balance_g, P, d, i, N, R, r)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
import warnings
warnings.filterwarnings('ignore')

def simulate_housing_decision_corrected(g, P, d, i, N, R, r, simulation_years=30):
    """
    修正版模拟：租金随房价增长而变化，保持租售比恒定
    """
    months = simulation_years * 12
    monthly_rate_house = g / 12  # 房价月增长率
    monthly_rate_invest = r / 12  # 投资月收益率
    
    # 计算固定月供
    monthly_payment = calculate_monthly_payment(P, d, i, N, 'equal_principal_interest')
    down_payment = P * d  # 首付金额
    
    # 创建数据存储
    monthly_data = pd.DataFrame({
        'month': range(1, months + 1),
        'house_value': np.zeros(months),  # 房屋价值
        'monthly_rent': np.zeros(months),  # 月租金（随房价变化）
        'monthly_investment': np.zeros(months),  # 每月可投资额
        'investment_balance': np.zeros(months),  # 投资账户余额
        'cumulative_investment': np.zeros(months)  # 累计投资额
    })
    
    # 逐月模拟
    investment_balance = down_payment  # 初始投资额 = 首付金额
    cumulative_investment = down_payment  # 累计投资额
    
    for month in range(months):
        # ===== 1. 更新房价和租金 =====
        current_house_value = P * (1 + monthly_rate_house) ** (month + 1)
        current_monthly_rent = current_house_value * R / 12  # 租金随房价增长
        
        # ===== 2. 计算当月可投资额 =====
        current_monthly_investment = monthly_payment - current_monthly_rent
        
        # ===== 3. 更新投资账户 =====
        investment_balance = investment_balance * (1 + monthly_rate_invest)
        investment_balance += current_monthly_investment
        cumulative_investment += abs(current_monthly_investment)
        
        # ===== 4. 记录数据 =====
        monthly_data.loc[month, 'house_value'] = current_house_value
        monthly_data.loc[month, 'monthly_rent'] = current_monthly_rent
        monthly_data.loc[month, 'monthly_investment'] = current_monthly_investment
        monthly_data.loc[month, 'investment_balance'] = investment_balance
        monthly_data.loc[month, 'cumulative_investment'] = cumulative_investment
    
    # 30年后的最终资产
    house_asset = monthly_data.iloc[-1]['house_value']  # 买房者净资产
    rent_asset = monthly_data.iloc[-1]['investment_balance']  # 租房者投资总额
    
    return house_asset, rent_asset, monthly_data

def find_balance_point_corrected(P, d, i, N, R, r, simulation_years=30):
    """
    寻找修正模型下的平衡点
    """
    def equation(g):
        house_asset, rent_asset, _ = simulate_housing_decision_corrected(
            g[0], P, d, i, N, R, r, simulation_years
        )
        return house_asset - rent_asset
    
    # 初始猜测：2%
    initial_guess = [0.02]
    
    try:
        solution = fsolve(equation, initial_guess, xtol=1e-8)
        balance_g = solution[0]
        
        # 验证结果
        house_asset, rent_asset, _ = simulate_housing_decision_corrected(
            balance_g, P, d, i, N, R, r, simulation_years
        )
        
        print(f" 修正版平衡点计算成功！")
        print(f"平衡点房价年增长率：{balance_g*100:.4f}%")
        print(f"30年后买房者资产：{house_asset:,.0f}元 ({house_asset/10000:.1f}万元)")
        print(f"30年后租房者资产：{rent_asset:,.0f}元 ({rent_asset/10000:.1f}万元)")
        print(f"资产差额：{house_asset - rent_asset:,.0f}元")
        
        if abs(house_asset - rent_asset) > 100000:  # 误差大于10万元
            print(f"  警告：平衡点验证误差较大，尝试二分法...")
            return find_balance_point_binary(P, d, i, N, R, r, simulation_years)
        
        return balance_g
    except Exception as e:
        print(f" 修正版平衡点计算失败：{e}")
        return find_balance_point_binary(P, d, i, N, R, r, simulation_years)

def find_balance_point_binary(P, d, i, N, R, r, simulation_years=30):
    """
    二分法寻找平衡点（备用方法）
    """
    low, high = 0.001, 0.10  # 0.1%到10%
    
    for iteration in range(100):  # 最多100次迭代
        mid = (low + high) / 2
        
        house_asset, rent_asset, _ = simulate_housing_decision_corrected(
            mid, P, d, i, N, R, r, simulation_years
        )
        
        diff = house_asset - rent_asset
        
        if iteration % 10 == 0:
            print(f"迭代{iteration:3d}: g={mid*100:6.3f}%, 差额={(diff/10000):8.2f}万元")
        
        # 收敛条件：差额小于1万元
        if abs(diff) < 10000:
            print(f"✅ 二分法收敛！迭代次数：{iteration}")
            return mid
        
        # 调整搜索范围
        if diff > 0:
            high = mid
        else:
            low = mid
    
    print(f"  二分法达到最大迭代次数，返回近似解")
    return (low + high) / 2

# 计算修正版平衡点
print("\n 寻找修正版买房-租房决策平衡点（租金随房价增长）...")
balance_g_corrected = find_balance_point_corrected(P, d, i, N, R, r, simulation_years)

# 测试不同的房价增长率
growth_rates = [0.005, 0.01, 0.015, balance_g_corrected, 0.025, 0.03, 0.04, 0.05]
results_corrected = []

print(f"\n 修正版多情景分析（不同房价年增长率）：")
print("-" * 100)

for g in growth_rates:
    house_asset, rent_asset, monthly_data = simulate_housing_decision_corrected(
        g, P, d, i, N, R, r, simulation_years
    )
    
    # 计算关键指标
    final_monthly_rent = monthly_data.iloc[-1]['monthly_rent']
    min_monthly_investment = monthly_data['monthly_investment'].min()
    negative_months = (monthly_data['monthly_investment'] < 0).sum()
    
    results_corrected.append({
        '房价年增长率': g * 100,
        '买房最终资产': house_asset,
        '租房最终资产': rent_asset,
        '资产差额': house_asset - rent_asset,
        '更优选择': '买房' if house_asset > rent_asset else '租房' if house_asset < rent_asset else '平衡点',
        '30年月租金': final_monthly_rent,
        '最低月现金流': min_monthly_investment,
        '负现金流月数': negative_months
    })
    
    print(f"g={g*100:6.3f}% | 买房: {house_asset/10000:8.1f}万元 | "
          f"租房: {rent_asset/10000:8.1f}万元 | "
          f"差额: {(house_asset-rent_asset)/10000:8.1f}万元 | "
          f"选择: {results_corrected[-1]['更优选择']} | "
          f"30年租金: {final_monthly_rent:6.0f}元")

# 创建结果DataFrame
results_df_corrected = pd.DataFrame(results_corrected)
print("\n修正版多情景分析汇总：")
print(results_df_corrected.to_string(index=False,
    float_format=lambda x: f"{x:.2f}" if isinstance(x, float) and x < 1 else f"{x:,.0f}",
    formatters={
        '房价年增长率': '{:.3f}%'.format,
        '买房最终资产': lambda x: f"{x/10000:.1f}万元",
        '租房最终资产': lambda x: f"{x/10000:.1f}万元",
        '资产差额': lambda x: f"{x/10000:.1f}万元",
        '30年月租金': lambda x: f"{x:.0f}元",
        '最低月现金流': lambda x: f"{x:.0f}元"
    }))

# 在平衡点进行详细模拟
print(f"\n 详细模拟（修正版平衡点g={balance_g_corrected*100:.4f}%）...")
house_asset_balance, rent_asset_balance, monthly_data_balance = simulate_housing_decision_corrected(
    balance_g_corrected, P, d, i, N, R, r, simulation_years
)

# 添加年份列
monthly_data_balance['year'] = (monthly_data_balance['month'] - 1) // 12 + 1
monthly_data_balance['month_in_year'] = (monthly_data_balance['month'] - 1) % 12 + 1

# 按年汇总
annual_data = monthly_data_balance.groupby('year').last().reset_index()
annual_data = annual_data[['year', 'house_value', 'monthly_rent', 'monthly_investment', 'investment_balance']]

print("\n 修正版年度资产变化（平衡点情景）：")
print(annual_data.head(15).to_string(index=False,
    formatters={
        'house_value': lambda x: f"{x/10000:.1f}万元",
        'monthly_rent': lambda x: f"{x:.0f}元",
        'monthly_investment': lambda x: f"{x:.0f}元",
        'investment_balance': lambda x: f"{x/10000:.1f}万元"
    }))

# 关键转折点分析
turning_point_month = monthly_data_balance[monthly_data_balance['monthly_investment'] < 0]['month'].min()
if pd.notna(turning_point_month):
    turning_point_year = (turning_point_month - 1) // 12 + 1
    turning_point_month_in_year = (turning_point_month - 1) % 12 + 1
    print(f"\n 关键转折点：第{turning_point_year}年{turning_point_month_in_year}月")
    print(f"   - 月租金首次超过月供：{monthly_data_balance.loc[turning_point_month-1, 'monthly_rent']:.0f}元 > {monthly_payment:,.0f}元")
    print(f"   - 月现金流首次转负：{monthly_data_balance.loc[turning_point_month-1, 'monthly_investment']:.0f}元")
else:
    print("\n30年内月现金流始终保持为正")

# 创建专业图表
fig = plt.figure(figsize=(16, 12))
fig.suptitle(f'上海"买房-租房"决策分析（修正版）\n'
             f'参数：总价{P/10000:.0f}万元，首付{d*100:.0f}%，贷款利率{i*100:.1f}%，'
             f'租售比{R*100:.1f}%，投资收益率{r*100:.1f}%\n'
             f'修正要点：租金随房价增长，保持租售比恒定 | 平衡点：{balance_g_corrected*100:.2f}%', 
             fontsize=14, fontweight='bold', y=0.98)

ax1 = plt.subplot(2, 2, 1)
years = annual_data['year']
ax1.plot(years, annual_data['house_value']/10000, 'o-', linewidth=2, markersize=6, 
         label='买房资产', color='blue', zorder=3)
ax1.plot(years, annual_data['investment_balance']/10000, 's-', linewidth=2, markersize=6, 
         label='租房资产', color='red', zorder=3)

# 标记关键点
final_house = annual_data.iloc[-1]['house_value']/10000
final_rent = annual_data.iloc[-1]['investment_balance']/10000
ax1.text(30, final_house*0.95, f'{final_house:.0f}万元', ha='right', va='bottom', fontsize=9, color='blue', fontweight='bold')
ax1.text(30, final_rent*0.95, f'{final_rent:.0f}万元', ha='right', va='bottom', fontsize=9, color='red', fontweight='bold')

# 标记转折点
if pd.notna(turning_point_month):
    turning_year = turning_point_year
    turning_house = monthly_data_balance.loc[turning_point_month-1, 'house_value']/10000
    turning_rent = monthly_data_balance.loc[turning_point_month-1, 'investment_balance']/10000
    ax1.axvline(x=turning_year, color='purple', linestyle='--', alpha=0.7, 
                label=f'租金>月供 ({turning_year}年)', zorder=1)
    ax1.scatter([turning_year], [turning_house], color='purple', s=80, zorder=4)
    ax1.scatter([turning_year], [turning_rent], color='purple', s=80, zorder=4)

ax1.set_xlabel('年份', fontsize=10)
ax1.set_ylabel('资产价值 (万元)', fontsize=10)
ax1.set_title(f'平衡点({balance_g_corrected*100:.2f}%)情景下资产增长轨迹', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# 月租金与月供对比
ax2 = plt.subplot(2, 2, 2)
years_plot = np.arange(1, 31)

# 年平均值
annual_rent = monthly_data_balance.groupby('year')['monthly_rent'].mean()
annual_investment = monthly_data_balance.groupby('year')['monthly_investment'].mean()

ax2.plot(years_plot, [monthly_payment]*30, 'k-', linewidth=2.5, label='固定月供', zorder=3)
ax2.plot(years_plot, annual_rent, 'r-', linewidth=2, label='年平均月租金', zorder=3)
ax2.fill_between(years_plot, 0, annual_rent, alpha=0.2, color='red', label='租金区域')

# 标记交叉点
if pd.notna(turning_point_month):
    ax2.axvline(x=turning_point_year, color='purple', linestyle='--', alpha=0.7,
                label=f'租金>月供 ({turning_point_year}年)', zorder=1)
    ax2.scatter([turning_point_year], [monthly_payment], color='purple', s=80, zorder=4)

ax2.set_xlabel('年份', fontsize=10)
ax2.set_ylabel('金额 (元)', fontsize=10)
ax2.set_title('月供 vs 月租金变化轨迹', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)
ax2.set_xlim(1, 30)

# 不同增长率下的最终资产对比
ax3 = plt.subplot(2, 2, 3)
x = results_df_corrected['房价年增长率']
width = 0.4
bars1 = ax3.bar(x - width/2, results_df_corrected['买房最终资产']/10000, width, 
                label='买房资产', alpha=0.8, color='skyblue', edgecolor='navy')
bars2 = ax3.bar(x + width/2, results_df_corrected['租房最终资产']/10000, width, 
                label='租房资产', alpha=0.8, color='salmon', edgecolor='maroon')

# 标记修正版平衡点
ax3.axvline(x=balance_g_corrected*100, color='green', linestyle='-', linewidth=2, 
            alpha=0.8, label=f'修正平衡点: {balance_g_corrected*100:.2f}%', zorder=2)

ax3.set_xlabel('房价年增长率 (%)', fontsize=10)
ax3.set_ylabel('30年后资产 (万元)', fontsize=10)
ax3.set_title('修正版：不同房价增长率下的最终资产对比', fontsize=12, fontweight='bold')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

# 在柱子上显示数值
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    height1 = bar1.get_height()
    height2 = bar2.get_height()
    if height1 > 0 and height1 < ax3.get_ylim()[1]*0.9:
        ax3.text(bar1.get_x() + bar1.get_width()/2., height1 + 20,
                 f'{height1:.0f}', ha='center', va='bottom', fontsize=8, rotation=90)
    if height2 > 0 and height2 < ax3.get_ylim()[1]*0.9:
        ax3.text(bar2.get_x() + bar2.get_width()/2., height2 + 20,
                 f'{height2:.0f}', ha='center', va='bottom', fontsize=8, rotation=90)

# 现金流变化分析
ax4 = plt.subplot(2, 2, 4)
ax4.plot(years_plot, annual_investment, 'g-', linewidth=2, markersize=6, 
         marker='D', markevery=2, label='年平均月现金流', zorder=3)

# 填充正负区域
positive_years = years_plot[annual_investment.values >= 0]
negative_years = years_plot[annual_investment.values < 0]

if len(positive_years) > 0:
    ax4.fill_between(positive_years, 0, annual_investment[positive_years.index].values, 
                    alpha=0.3, color='green', label='正现金流', zorder=1)
if len(negative_years) > 0:
    ax4.fill_between(negative_years, 0, annual_investment[negative_years.index].values, 
                    alpha=0.3, color='red', label='负现金流', zorder=1)

# 标记零线
ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5, zorder=2)

# 标记转折点
if pd.notna(turning_point_month):
    ax4.axvline(x=turning_point_year, color='purple', linestyle='--', alpha=0.7,
                label=f'现金流转折 ({turning_point_year}年)', zorder=4)
    ax4.scatter([turning_point_year], [0], color='purple', s=80, zorder=5)

ax4.set_xlabel('年份', fontsize=10)
ax4.set_ylabel('月现金流 (元)', fontsize=10)
ax4.set_title('租房者月现金流变化（月供 - 月租金）', fontsize=12, fontweight='bold')
ax4.legend(loc='upper right', fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(1, 30)
ax4.set_ylim(annual_investment.min()*1.2, max(annual_investment.max()*1.2, 20000))

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('shanghai_housing_analysis_corrected.png', dpi=300, bbox_inches='tight')
plt.show()


print(f"\n" + "="*80)

print(f"\n上海市场关键参数（修正版）：")
print(f"- 房屋总价：{P/10000:.0f}万元 ({house_area}㎡，{house_price_per_sqm/10000:.2f}万元/㎡)")
print(f"- 初始月供：{monthly_payment:,.0f}元，初始月租金：{initial_monthly_rent:,.0f}元")
print(f"- 租售比：{R*100:.1f}%（保持恒定，租金随房价增长）")

print(f"\n修正版决策平衡点：")
print(f"- 临界房价年增长率：{balance_g_corrected*100:.4f}%")
print(f"- 30年后平衡资产：{house_asset_balance/10000:.1f}万元")

print(f"\n关键动态变化：")
if pd.notna(turning_point_month):
    print(f"- 第{turning_point_year}年{turning_point_month_in_year}月：租金首次超过月供")
    print(f"  • 30年月租金：{monthly_data_balance.iloc[-1]['monthly_rent']:,.0f}元")
    print(f"  • 30年月供：{monthly_payment:,.0f}元（固定不变）")
    print(f"- 前{turning_point_year-1}年：月现金流为正，可积累投资")
    print(f"- 后{30-turning_point_year+1}年：月现金流为负，需额外投入")
else:
    print("- 30年内租金始终低于月供，月现金流始终保持为正")

print(f"\n上海市场现实意义：")
print(f"- 2025年上海房价同比下跌3.59%，但长期历史平均增长率约5-6%")
print(f"- 修正模型显示：当增长率>5%时，买房资产({results_df_corrected.iloc[-1]['买房最终资产']/10000:.1f}万元)远超租房资产({results_df_corrected.iloc[-1]['租房最终资产']/10000:.1f}万元)")
print(f"- 关键启示：长期持有房产在通胀环境下具有显著的财务优势")


In [ ]:
def analyze_interest_rate_sensitivity(P, d, N, R, r, g_initial=0.02, simulation_years=30):
    """
    分析贷款利率对平衡点的影响
    """
    interest_rates = np.linspace(0.03, 0.07, 9)  # 从3%到7%
    balance_points = []
    monthly_payments = []
    
    print("贷款利率敏感性分析：")
    print("-" * 80)
    print(f"{'贷款利率':^10} | {'月供(元)':^12} | {'平衡点增长率':^15} | {'30年买房资产':^15} | {'30年租房资产':^15}")
    print("-" * 80)
    
    for i in interest_rates:
        # 计算月供
        monthly_payment = calculate_monthly_payment(P, d, i, N, 'equal_principal_interest')
        monthly_payments.append(monthly_payment)
        
        # 寻找平衡点
        def equation(g):
            house_asset, rent_asset, _ = simulate_housing_decision(
                g[0], P, d, i, N, R, r, simulation_years
            )
            return house_asset - rent_asset
        
        try:
            solution = fsolve(equation, [g_initial], xtol=1e-6)
            balance_g = solution[0]
            balance_points.append(balance_g)
            
            # 验证结果
            house_asset, rent_asset, _ = simulate_housing_decision(
                balance_g, P, d, i, N, R, r, simulation_years
            )
            
            print(f"{i*100:8.2f}% | {monthly_payment:12,.0f} | {balance_g*100:13.4f}% | "
                  f"{house_asset/10000:13.1f}万元 | {rent_asset/10000:13.1f}万元")
            
        except Exception as e:
            print(f"利率{i*100:.2f}%计算失败: {e}")
            balance_points.append(np.nan)
    
    return interest_rates, balance_points, monthly_payments

# 执行敏感性分析
interest_rates, balance_points, monthly_payments = analyze_interest_rate_sensitivity(
    P, d, N, R, r, balance_g, simulation_years
)

In [ ]:
# 创建专业可视化
plt.figure(figsize=(12, 6))

# 图1：利率 vs 平衡点
ax1 = plt.gca()
ax1.plot(interest_rates*100, [g*100 for g in balance_points], 'bo-', linewidth=2.5, markersize=8,
         color='blue', label='平衡点房价年增长率')
ax1.set_xlabel('贷款年利率 (%)', fontsize=12)
ax1.set_ylabel('平衡点房价年增长率 (%)', fontsize=12, color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True, alpha=0.3)

# 添加数据标签
for i, (rate, balance_g) in enumerate(zip(interest_rates, balance_points)):
    ax1.text(rate*100, balance_g*100 + 0.1, f'{balance_g*100:.2f}%', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# 图2：利率 vs 月供（次Y轴）
ax2 = ax1.twinx()
ax2.plot(interest_rates*100, monthly_payments, 'rs-', linewidth=2.5, markersize=8,
         color='red', label='月供金额')
ax2.set_ylabel('月供金额 (元)', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('贷款利率对买房-租房决策平衡点的影响（上海市场）\n'
          '参数：总价440万元，首付30%，租售比1.8%，投资收益率4.5%', 
          fontsize=14, fontweight='bold', pad=20)
plt.legend(['平衡点房价年增长率', '月供金额'], loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# 上海核心参数
P = 85000 * 80  # 总价440万元
d = 0.3  # 首付比例30%
i = 0.04  # 贷款利率4.0%
N = 30  # 贷款年限
R = 0.018  # 租售比1.8%
fixed_rent = P * R / 12  # 固定月租金

# 计算月供（等额本息）
def calculate_monthly_payment(P, d, i, N):
    loan_amount = P * (1 - d)
    monthly_rate = i / 12
    total_months = N * 12
    return loan_amount * monthly_rate * (1 + monthly_rate)**total_months / ((1 + monthly_rate)**total_months - 1)

monthly_payment = calculate_monthly_payment(P, d, i, N)
monthly_cashflow = monthly_payment - fixed_rent  # 固定月现金流

print(f"核心参数（简化模型）：")
print(f"房屋总价：{P/10000:.0f}万元")
print(f"等额本息月供：{monthly_payment:,.0f}元")
print(f"固定月租金：{fixed_rent:,.0f}元")
print(f"固定月现金流（可投资额）：{monthly_cashflow:,.0f}元")


def find_balance_point(P, d, i, N, fixed_rent, r, simulation_years=30, initial_guess=None):
    """
    优化版平衡点计算，使用动态初始猜测
    """
    monthly_payment = calculate_monthly_payment(P, d, i, N)
    down_payment = P * d
    monthly_rate_invest = r / 12
    
    # 动态初始猜测：基于投资收益率调整
    if initial_guess is None:
        # 基础猜测：投资收益率的70%
        base_guess = r * 0.7
        # 确保在合理范围内
        initial_guess = max(0.01, min(0.10, base_guess))
    
    def equation(g):
        # 买房者30年后资产
        house_asset = P * (1 + g) ** simulation_years
        
        # 租房者30年后资产
        monthly_rate_house = g / 12
        total_months = simulation_years * 12
        
        # 首付的复利增长
        down_payment_growth = down_payment * (1 + monthly_rate_invest) ** total_months
        
        # 月现金流的复利增长（年金终值公式）
        if abs(monthly_rate_invest) < 1e-10:  # 避免除零错误
            cashflow_growth = monthly_cashflow * total_months
        else:
            cashflow_growth = monthly_cashflow * ((1 + monthly_rate_invest) ** total_months - 1) / monthly_rate_invest
        
        rent_asset = down_payment_growth + cashflow_growth
        
        return house_asset - rent_asset
    
    # 使用fsolve寻找平衡点
    solution = fsolve(equation, initial_guess, xtol=1e-8)
    balance_g = solution[0]
    
    # 验证结果合理性
    if balance_g < 0 or balance_g > 0.20:  # 限制在0-20%范围内
        print(f" 警告：平衡点{balance_g*100:.2f}%超出合理范围，尝试调整初始猜测")
        # 尝试不同的初始猜测
        for guess in [0.02, 0.05, 0.08]:
            solution = fsolve(equation, guess, xtol=1e-8)
            balance_g = solution[0]
            if 0 <= balance_g <= 0.20:
                break
    
    return balance_g

# 分析不同投资收益率下的平衡点
return_rates = np.linspace(0.04, 0.15, 12)  # 4%到15%
balance_points = []
initial_guess = 0.03  # 初始猜测

print("投资收益率敏感性分析（简化模型）：")
print("-" * 60)
print(f"{'投资收益率':^12} | {'平衡点增长率':^15} | {'状态':^8}")
print("-" * 60)

for r in return_rates:
    balance_g = find_balance_point(P, d, i, N, fixed_rent, r, initial_guess=initial_guess)
    balance_points.append(balance_g)
    
    status = "合理" if 0 <= balance_g <= 0.20 else "❌ 异常"
    print(f"{r*100:10.1f}% | {balance_g*100:13.4f}% | {status:^8}")
    
    # 为下一次计算设置更好的初始猜测
    initial_guess = balance_g * 0.9  # 略低于当前平衡点


fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('高投资能力对买房-租房决策平衡点的影响（简化模型）\n租金固定不变，专注投资收益率变化', 
             fontsize=16, fontweight='bold', y=0.98)

# 图1：投资收益率 vs 平衡点
ax1.plot(return_rates*100, [g*100 for g in balance_points], 'bo-', linewidth=2.5, markersize=8,
        color='blue', label='平衡点房价年增长率')
ax1.set_xlabel('投资年化收益率 (%)', fontsize=12)
ax1.set_ylabel('平衡点房价年增长率 (%)', fontsize=12)
ax1.set_title('投资能力与平衡点关系', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# 添加数据标签
for r, g in zip(return_rates, balance_points):
    ax1.text(r*100, g*100 + 0.1, f'{g*100:.1f}%', ha='center', va='bottom', fontsize=9)

# 图2：关键情景对比（r=4.5% vs r=10%）
r_normal = 0.045
r_high = 0.10
g_normal = find_balance_point(P, d, i, N, fixed_rent, r_normal)
g_high = find_balance_point(P, d, i, N, fixed_rent, r_high)

years = np.arange(1, 31)
house_normal = P * (1 + g_normal) ** years
house_high = P * (1 + g_high) ** years

# 租房者资产计算函数
def calculate_rent_asset(r, simulation_years=30):
    down_payment = P * d
    monthly_rate_invest = r / 12
    total_months = simulation_years * 12
    
    # 首付复利
    down_payment_growth = down_payment * (1 + monthly_rate_invest) ** total_months
    
    # 月现金流复利
    if abs(monthly_rate_invest) < 1e-10:
        cashflow_growth = monthly_cashflow * total_months
    else:
        cashflow_growth = monthly_cashflow * ((1 + monthly_rate_invest) ** total_months - 1) / monthly_rate_invest
    
    return down_payment_growth + cashflow_growth

# 计算年度资产
rent_normal_yearly = []
rent_high_yearly = []
for year in years:
    # 简化计算：按年复利近似
    down_payment_growth_normal = P * d * (1 + r_normal) ** year
    cashflow_growth_normal = monthly_cashflow * 12 * ((1 + r_normal) ** year - 1) / r_normal
    rent_normal_yearly.append(down_payment_growth_normal + cashflow_growth_normal)
    
    down_payment_growth_high = P * d * (1 + r_high) ** year
    cashflow_growth_high = monthly_cashflow * 12 * ((1 + r_high) ** year - 1) / r_high
    rent_high_yearly.append(down_payment_growth_high + cashflow_growth_high)

ax2.plot(years, house_normal/10000, 'b-', linewidth=2.5, label=f'买房 (r=4.5%, g={g_normal*100:.2f}%)')
ax2.plot(years, rent_normal_yearly/np.array(10000), 'b--', linewidth=2.5, label=f'租房 (r=4.5%)')
ax2.plot(years, house_high/10000, 'r-', linewidth=2.5, label=f'买房 (r=10%, g={g_high*100:.2f}%)')
ax2.plot(years, rent_high_yearly/np.array(10000), 'r--', linewidth=2.5, label=f'租房 (r=10%)')

ax2.set_xlabel('年份', fontsize=12)
ax2.set_ylabel('资产价值 (万元)', fontsize=12)
ax2.set_title('不同投资能力下的资产增长轨迹', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

# 图3：平衡点移动的动态关系
# 计算在固定房价增长率(5%)下，不同投资收益率的资产差额
fixed_g = 0.05
asset_diffs = []

for r in return_rates:
    # 买房资产
    house_asset = P * (1 + fixed_g) ** 30
    
    # 租房资产
    monthly_rate_invest = r / 12
    total_months = 30 * 12
    
    down_payment_growth = (P * d) * (1 + monthly_rate_invest) ** total_months
    if abs(monthly_rate_invest) < 1e-10:
        cashflow_growth = monthly_cashflow * total_months
    else:
        cashflow_growth = monthly_cashflow * ((1 + monthly_rate_invest) ** total_months - 1) / monthly_rate_invest
    
    rent_asset = down_payment_growth + cashflow_growth
    asset_diff = house_asset - rent_asset  # 正值表示买房更优
    asset_diffs.append(asset_diff / 10000)  # 万元

ax3.plot(return_rates*100, asset_diffs, 'k-', linewidth=2.5, color='darkblue')
ax3.fill_between(return_rates*100, asset_diffs, 0, 
                where=(np.array(asset_diffs) > 0), color='green', alpha=0.3,
                label='买房更优')
ax3.fill_between(return_rates*100, asset_diffs, 0, 
                where=(np.array(asset_diffs) < 0), color='red', alpha=0.3,
                label='租房更优')
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.7)
ax3.set_xlabel('投资年化收益率 (%)', fontsize=12)
ax3.set_ylabel(f'30年资产差额 (买房-租房, 万元)\n(房价增长率={fixed_g*100:.0f}%)', fontsize=12)
ax3.set_title('投资能力与资产差异关系', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

# 图4：平衡点移动速度分析
# 计算平衡点对投资收益率的敏感度
sensitivity = np.gradient(balance_points, return_rates)
ax4.plot(return_rates*100, sensitivity, 'm-', linewidth=2.5, markersize=8,
        color='purple', label='平衡点变化速度')
ax4.set_xlabel('投资年化收益率 (%)', fontsize=12)
ax4.set_ylabel('平衡点变化速度\n(%房价增长率/%投资收益率)', fontsize=12)
ax4.set_title('平衡点移动速度分析', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# 标记关键区域
ax4.axvspan(4, 6, alpha=0.2, color='green', label='保守投资者')
ax4.axvspan(8, 12, alpha=0.2, color='yellow', label='积极投资者')
ax4.axvspan(12, 15, alpha=0.2, color='red', label='专业投资者')
ax4.legend(loc='upper left', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('investment_return_sensitivity_simple.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import akshare as ak
import numpy as np

# 2. 数据获取函数
def get_stock_data(stock_code, start_date, end_date):
    """
    获取单只股票的历史数据
    """
    try:
        # AKShare获取A股历史数据
        df = ak.stock_zh_a_hist(symbol=stock_code, period="daily", 
                               start_date=start_date, end_date=end_date,
                               adjust="hfq")  # 后复权
        df['date'] = pd.to_datetime(df['日期'])
        df.set_index('date', inplace=True)
        return df[['收盘']]
    except Exception as e:
        print(f"获取股票 {stock_code} 数据失败: {e}")
        return None

def get_index_data(index_code, start_date, end_date):
    """
    获取指数历史数据
    """
    try:
        # AKShare获取指数数据
        df = ak.index_zh_a_hist(symbol=index_code, period="daily",
                               start_date=start_date, end_date=end_date)
        df['date'] = pd.to_datetime(df['日期'])
        df.set_index('date', inplace=True)
        return df[['收盘']]
    except Exception as e:
        print(f"获取指数 {index_code} 数据失败: {e}")
        return None

# 3. 回测参数设置
start_date = "20100101"
end_date = datetime.datetime.now().strftime("%Y%m%d")
initial_capital = 100000  # 初始资金10万元
annual_investment = 20000  # 每年追加2万元
tickers = {
    '600519': '贵州茅台',
    '600036': '招商银行', 
    '601318': '中国平安'
}
benchmark_code = "000300"  # 沪深300

# 4. 获取数据
print("正在获取股票数据...")
data_dict = {}
for code, name in tickers.items():
    print(f"获取 {name}({code}) 数据...")
    stock_data = get_stock_data(code, start_date, end_date)
    if stock_data is not None:
        data_dict[code] = stock_data['收盘']
        print(f"{name} 数据获取成功，共 {len(stock_data)} 条记录")

print("正在获取基准指数数据...")
benchmark_data = get_index_data(benchmark_code, start_date, end_date)
if benchmark_data is not None:
    data_dict[benchmark_code] = benchmark_data['收盘']
    print(f"沪深300 数据获取成功，共 {len(benchmark_data)} 条记录")

# 合并数据
price_data = pd.DataFrame(data_dict)
price_data.columns = list(tickers.values()) + ['沪深300']
price_data = price_data.dropna()

print("数据获取完成！")
print(f"数据时间范围: {price_data.index.min()} 到 {price_data.index.max()}")
print(f"可用数据点: {len(price_data)}")

In [ ]:
price_data = pd.DataFrame(data_dict)
# 只需要确保沪深300的列名正确即可
price_data.rename(columns={'000300': '沪深300'}, inplace=True) # 假设000300是键名
price_data = price_data.dropna()
price_data

In [ ]:
yearly_prices = price_data.resample('Y').first().dropna()
years = yearly_prices.index.year.tolist()

# 股票代码列表，用于索引
ticker_codes = list(tickers.keys())
benchmark_name = price_data.columns[-1]

# 初始化回测结果存储 DataFrame
portfolio_metrics = pd.DataFrame(index=years)
portfolio_metrics['Strategy_Capital'] = 0.0      # 策略组合净值
portfolio_metrics['Benchmark_Capital'] = 0.0     # 基准组合净值 (包含定投)

# 初始化
current_strategy_capital = initial_capital
current_benchmark_capital = initial_capital
shares = pd.Series(0.0, index=ticker_codes) # 策略持股数
benchmark_shares = 0.0                      # 基准持股数


# 循环模拟每年的投资和调仓
for i in range(len(years)):
    current_year = years[i]
    
    # 获取当年年初的调仓日价格
    rebalance_date = yearly_prices.index[i]
    current_prices = yearly_prices.loc[rebalance_date]
    
    # 1. 年初追加投资 (除第一年外)
    if i > 0:
        current_strategy_capital += annual_investment
        current_benchmark_capital += annual_investment

    # 2. 策略组合 - 等权分配资金 (调仓)
    # 计算当前每只股票可分配的资金
    capital_per_stock = current_strategy_capital / len(ticker_codes)
    # 计算新的持股数
    shares = (capital_per_stock / current_prices[ticker_codes])
    
    # 3. 基准组合 - 购买沪深300 (调仓)
    benchmark_shares = current_benchmark_capital / current_prices[benchmark_name]
    
    # 4. 计算年末资产净值 (使用下一年的调仓日价格作为年末价格)
    if i < len(years) - 1:
        next_year = years[i+1]
        next_rebalance_date = yearly_prices.index[i+1]
        next_prices = yearly_prices.loc[next_rebalance_date]
        
        # 策略净值：持有股份 * 下一年年初价格
        end_of_year_strategy_value = (shares * next_prices[ticker_codes]).sum()
        current_strategy_capital = end_of_year_strategy_value
        portfolio_metrics.loc[current_year, 'Strategy_Capital'] = end_of_year_strategy_value
        
        # 基准净值：持有股份 * 下一年年初价格
        end_of_year_benchmark_value = benchmark_shares * next_prices[benchmark_name]
        current_benchmark_capital = end_of_year_benchmark_value
        portfolio_metrics.loc[current_year, 'Benchmark_Capital'] = end_of_year_benchmark_value
    else:
        # 回测期末的净值 (使用数据中最后一天价格)
        final_prices = price_data.iloc[-1]
        
        # 策略最终净值
        final_strategy_value = (shares * final_prices[ticker_codes]).sum()
        portfolio_metrics.loc[current_year, 'Strategy_Capital'] = final_strategy_value
        
        # 基准最终净值
        final_benchmark_value = benchmark_shares * final_prices[benchmark_name]
        portfolio_metrics.loc[current_year, 'Benchmark_Capital'] = final_benchmark_value


# 6. 关键指标计算 (年度收益率)

# 创建一个包含初始资金的 DataFrame 用于计算收益率
initial_df = pd.DataFrame({
    'Strategy_Capital': [initial_capital],
    'Benchmark_Capital': [initial_capital]
}, index=[years[0]-1])

# 合并初始资金和每年的年末净值
portfolio_final = pd.concat([initial_df, portfolio_metrics[['Strategy_Capital', 'Benchmark_Capital']]])
portfolio_final.index.name = 'Year'

# 计算年度收益率 (与上一年年末净值对比)
portfolio_metrics['Strategy_Annual_Return'] = portfolio_final['Strategy_Capital'].pct_change().shift(-1).loc[years]
portfolio_metrics['Benchmark_Annual_Return'] = portfolio_final['Benchmark_Capital'].pct_change().shift(-1).loc[years]

# 格式化百分比
portfolio_metrics['Strategy_Annual_Return (%)'] = portfolio_metrics['Strategy_Annual_Return'] * 100
portfolio_metrics['Benchmark_Annual_Return (%)'] = portfolio_metrics['Benchmark_Annual_Return'] * 100

# 7. 计算夏普比率 (基于日度收益率的年度化指标)

# 计算策略组合的每日净值变化（以等权组合的平均日收益率近似）
daily_returns = price_data.pct_change().dropna()
strategy_daily_return = daily_returns[ticker_codes].mean(axis=1)
benchmark_daily_return = daily_returns[benchmark_name]

# 假设无风险收益率为 3% (年化)
risk_free_rate = 0.03
trading_days = 252 # 每年约 252 个交易日

def calculate_sharpe(daily_returns, risk_free_rate, trading_days):
    """计算夏普比率 (年化)"""
    # 年化平均收益率 = 每日平均收益率 * 交易日
    annualized_return = daily_returns.mean() * trading_days
    # 年化波动率 = 每日收益率标准差 * sqrt(交易日)
    annualized_volatility = daily_returns.std() * np.sqrt(trading_days)
    
    if annualized_volatility == 0:
        return 0.0
    return (annualized_return - risk_free_rate) / annualized_volatility

sharpe_strategy = calculate_sharpe(strategy_daily_return, risk_free_rate, trading_days)
sharpe_benchmark = calculate_sharpe(benchmark_daily_return, risk_free_rate, trading_days)


# 8. 结果导出
file_name = f'实验_LLM策略回测结果_{ticker_codes[0]}_to_{years[-1]}.xlsx'

# 准备总览数据
total_years = len(years)
total_invested = initial_capital + annual_investment * (total_years - 1)
total_summary = pd.DataFrame({
    '指标': ['回测期', '总投入资金', '策略最终资产', '基准最终资产', '策略总收益率', '基准总收益率', '策略夏普比率', '基准夏普比率'],
    '数值': [
        f'{years[0]} - {years[-1]}',
        f'¥{total_invested:,.2f}',
        f'¥{portfolio_final["Strategy_Capital"].iloc[-1]:,.2f}',
        f'¥{portfolio_final["Benchmark_Capital"].iloc[-1]:,.2f}',
        f'{((portfolio_final["Strategy_Capital"].iloc[-1] / total_invested) - 1) * 100:.2f}%',
        f'{((portfolio_final["Benchmark_Capital"].iloc[-1] / total_invested) - 1) * 100:.2f}%',
        f'{sharpe_strategy:.3f}',
        f'{sharpe_benchmark:.3f}'
    ]
})

with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    # 资产净值数据
    portfolio_final.to_excel(writer, sheet_name='资产净值增长')
    
    # 年度收益率数据
    portfolio_metrics[['Strategy_Annual_Return (%)', 'Benchmark_Annual_Return (%)']].to_excel(
        writer, sheet_name='年度收益率', float_format='%.2f'
    )
    
    # 总结数据
    total_summary.to_excel(writer, sheet_name='回测总结', index=False)
    
print(f"\n 策略回测完成，数据已导出至 Excel 文件: {file_name}")

print("\n--- 关键结果摘要 ---")
print(f"回测期: {years[0]} - {years[-1]}")
print(f"策略最终资产: ¥{portfolio_final['Strategy_Capital'].iloc[-1]:,.2f}")
print(f"基准最终资产: ¥{portfolio_final['Benchmark_Capital'].iloc[-1]:,.2f}")
print(f"策略夏普比率: {sharpe_strategy:.3f}")
print(f"基准夏普比率: {sharpe_benchmark:.3f}")
print("--------------------")
